# SquirrelBGone — YOLOv8s Multi-Class Training (v2)

Trains a YOLOv8**s** model on the full Warren Wiens Squirrel Detector 1.1 dataset.
Produces a multi-class `best.pt` that detects squirrels **and** birds, so bird detections
can be suppressed in `detect.py` instead of misclassified as squirrels.

**Classes trained:** squirrel, bird, cat, dog, raccoon, skunk, deer

**Why YOLOv8s over n?**
The small model (~11M params) meaningfully improves multi-class precision over nano (~3M params)
at roughly the same inference time on Pi 5 CPU for this image size.
If inference is too slow after deployment, change `yolov8s.pt` → `yolov8n.pt` in Step 5.

**Before running:** Runtime → Change runtime type → T4 GPU

## Step 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — go to Runtime > Change runtime type > T4 GPU'}")

## Step 2 — Install dependencies

In [ ]:
%pip install -q ultralytics roboflow
print("Done.")

## Step 3 — Download dataset

Warren Wiens Squirrel Detector 1.1 — 5,102 images, 7 classes, open source.

Get a free Roboflow API key at https://app.roboflow.com/settings/api

In [ ]:
from roboflow import Roboflow
from pathlib import Path

ROBOFLOW_API_KEY = "rf_your_key_here"  # paste your key here

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("warren-wiens-d0d4p").project("squirrel-detector-1.1")
dataset = project.version(1).download("yolov8")

print(f"Dataset location: {dataset.location}")

## Step 4 — Inspect dataset and validate classes

In [ ]:
import yaml

data_yaml = Path(dataset.location) / "data.yaml"
with open(data_yaml) as f:
    data = yaml.safe_load(f)

print("Classes:", data["names"])
print("Num classes:", data["nc"])

# Confirm squirrel and bird are both present — required for false-positive suppression
required = {"squirrel", "bird"}
present  = set(data["names"])
missing  = required - present
if missing:
    raise ValueError(f"Required classes missing from dataset: {missing}")
print("\n✓ squirrel and bird both present")

# Split sizes
for split in ["train", "valid", "test"]:
    img_dir = Path(dataset.location) / split / "images"
    if img_dir.exists():
        print(f"{split}: {len(list(img_dir.glob('*.*')))} images")

## Step 5 — Train YOLOv8s

- 75 epochs with patience=15 early stopping
- `batch=16` is safe for T4; try `batch=32` if no OOM
- Expect ~35–45 minutes on T4

To fall back to the faster nano model, change `yolov8s.pt` → `yolov8n.pt` below.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # small — better multi-class accuracy than nano

results = model.train(
    data=str(data_yaml),
    epochs=75,
    imgsz=640,
    batch=16,
    name="squirrelbgone_v2",
    patience=15,
    save=True,
    plots=True,
    device=0,
    workers=2,
)

print("\nTraining complete.")
print(f"Best weights: {results.save_dir}/weights/best.pt")

## Step 6 — Evaluate on test set

In [ ]:
best_model = YOLO(f"{results.save_dir}/weights/best.pt")

metrics = best_model.val(
    data=str(data_yaml),
    split="test",
)

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

# Per-class breakdown
print("\nPer-class AP50:")
per_class = dict(zip(data["names"], metrics.box.ap50))
for name, ap in sorted(per_class.items(), key=lambda x: -x[1]):
    print(f"  {name:<12} {ap:.3f}")

# Explicit check on the two classes we care about most
print("\n--- Key classes ---")
for cls in ["squirrel", "bird"]:
    ap = per_class.get(cls, 0)
    flag = "✓" if ap >= 0.70 else "⚠ low"
    print(f"  {cls:<12} AP50={ap:.3f}  {flag}")

## Step 7 — Download best.pt

In [ ]:
import shutil
from google.colab import files

best_pt = f"{results.save_dir}/weights/best.pt"
dest    = "/content/squirrelbgone_v2_best.pt"
shutil.copy(best_pt, dest)

print(f"Model size: {Path(best_pt).stat().st_size / 1e6:.1f} MB")
print("Downloading...")
files.download(dest)

## Step 8 — Smoke test (optional)

Run a few test images through the model and confirm squirrel and bird both appear in predictions.

In [ ]:
import glob
from IPython.display import Image, display

test_images = glob.glob(f"{dataset.location}/test/images/*.*")[:4]
if not test_images:
    print("No test images found.")
else:
    for img_path in test_images:
        result = best_model.predict(img_path, conf=0.35, save=True,
                                    project="/content", name="smoke_test",
                                    exist_ok=True)
        annotated = (glob.glob("/content/smoke_test/*.jpg") +
                     glob.glob("/content/smoke_test/*.png"))
        if annotated:
            display(Image(sorted(annotated)[-1], width=640))
        for r in result:
            for box in r.boxes:
                cls  = data["names"][int(box.cls)]
                conf = float(box.conf)
                print(f"  {cls}: {conf:.2f}")
        print()

## Deployment

1. Move the downloaded file into the repo: `models/squirrelbgone_v2_best.pt`
2. Commit and push
3. On the Pi: `git pull`
4. In `.env`, set: `MODEL_PATH=models/squirrelbgone_v2_best.pt`
5. Restart `detect.py`

The multi-class model will now log `class=bird` for bird detections.
`BENIGN_CLASSES` in `detect.py` suppresses them from triggering the sprayer.

**If inference is too slow on Pi 5** (check the log — it prints fps on startup), retrain
with `yolov8n.pt` in Step 5. The accuracy tradeoff is small for a well-annotated dataset
this size.